# Left/Right Hand YOLO Video Demonstration

Render the trained EgoHands left/right YOLO detections back onto a world-camera video.

In [ ]:
from pathlib import Path
import csv
import random
from collections import Counter, defaultdict

import cv2
from IPython.display import Video, display

DATASET_ROOT = Path("/home/julius/datasets/seminar_object_detection")
EXPORT_ROOT = DATASET_ROOT / "iAssistADL_dataset_export_2026-05-18"
VIDEO_ROOT = EXPORT_ROOT / "world_camera_videos"
YOLO_LEFT_RIGHT_ROOT = DATASET_ROOT / "lower_feature_outcome" / "yolo_left_right_hand_outcome_csv"
OUTPUT_ROOT = DATASET_ROOT / "video_demonstration" / "left_right_hand_yolo"

# Set this to a list of full_id strings to force specific videos, or keep None to randomly sample videos.
SELECTED_FULL_IDS = None
RANDOM_SEED = 42
RANDOM_VIDEO_COUNT = 5

RENDER_FULL_VIDEO = True
WINDOW_SECONDS = 30
MAX_BOXES_PER_FRAME = 8
MIN_CONFIDENCE = 0.25

CLASS_COLORS = {
    "left_hand": (70, 220, 90),
    "right_hand": (255, 130, 70),
}
DEFAULT_COLOR = (40, 220, 255)


def participant_from_full_id(full_id):
    return "_".join(full_id.split("_")[:4])


def list_detection_csvs():
    return sorted(YOLO_LEFT_RIGHT_ROOT.glob("*_world_camera.csv"))


def read_detections(path, min_confidence=0.0):
    detections_by_frame = defaultdict(list)
    with open(path, newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            confidence = float(row["confidence"])
            if confidence < min_confidence:
                continue
            frame_id = int(float(row["frame_id"]))
            detections_by_frame[frame_id].append(
                {
                    "class_name": row.get("class_name", "hand"),
                    "confidence": confidence,
                    "x1": float(row["x1"]),
                    "y1": float(row["y1"]),
                    "x2": float(row["x2"]),
                    "y2": float(row["y2"]),
                }
            )
    return detections_by_frame


def detection_summary(csv_path):
    detections_by_frame = read_detections(csv_path, MIN_CONFIDENCE)
    class_counts = Counter()
    for detections in detections_by_frame.values():
        for det in detections:
            class_counts[det["class_name"]] += 1
    return {
        "csv_path": csv_path,
        "full_id": csv_path.name.replace("_world_camera.csv", ""),
        "frames_with_detection": len(detections_by_frame),
        "detections": sum(class_counts.values()),
        "left_hand": class_counts.get("left_hand", 0),
        "right_hand": class_counts.get("right_hand", 0),
    }


csv_paths = list_detection_csvs()
if not csv_paths:
    raise FileNotFoundError(f"No left/right hand YOLO CSV files found yet: {YOLO_LEFT_RIGHT_ROOT}")

summaries = [detection_summary(path) for path in csv_paths]
summary_by_full_id = {row["full_id"]: row for row in summaries}

valid_summaries = []
for row in summaries:
    full_id = row["full_id"]
    participant_id = participant_from_full_id(full_id)
    video_path = VIDEO_ROOT / participant_id / f"{full_id}_world_camera.mp4"
    if video_path.exists():
        valid_summaries.append(row)

if SELECTED_FULL_IDS is None:
    rng = random.Random(RANDOM_SEED)
    selected_summaries = rng.sample(valid_summaries, k=min(RANDOM_VIDEO_COUNT, len(valid_summaries)))
else:
    selected_summaries = []
    for full_id in SELECTED_FULL_IDS:
        if full_id not in summary_by_full_id:
            available = "\n".join(f"  {row['full_id']}" for row in summaries[:50])
            raise FileNotFoundError(f"No CSV for full_id={full_id}. Available examples:\n{available}")
        selected_summaries.append(summary_by_full_id[full_id])

duration_label = "full_video" if RENDER_FULL_VIDEO else f"{WINDOW_SECONDS}s"
selected_records = []
for selected in selected_summaries:
    full_id = selected["full_id"]
    participant_id = participant_from_full_id(full_id)
    video_path = VIDEO_ROOT / participant_id / f"{full_id}_world_camera.mp4"
    detections_path = YOLO_LEFT_RIGHT_ROOT / f"{full_id}_world_camera.csv"
    output_video_path = OUTPUT_ROOT / f"{full_id}_left_right_hand_yolo_demo_{duration_label}.mp4"
    for path in [video_path, detections_path]:
        if not path.exists():
            raise FileNotFoundError(path)
    selected_records.append(
        {
            "full_id": full_id,
            "video_path": video_path,
            "detections_path": detections_path,
            "output_video_path": output_video_path,
            "summary": selected,
        }
    )

[
    {
        "full_id": row["full_id"],
        "video_path": str(row["video_path"]),
        "detections_path": str(row["detections_path"]),
        "output_video_path": str(row["output_video_path"]),
        "detections": row["summary"]["detections"],
        "frames_with_detection": row["summary"]["frames_with_detection"],
    }
    for row in selected_records
]

## Pick The Render Window

In [ ]:
def get_video_info(video_path):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {video_path}")
    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()
    return fps, frame_count, width, height


render_jobs = []
for record in selected_records:
    fps, frame_count, width, height = get_video_info(record["video_path"])
    detections_by_frame = read_detections(record["detections_path"], MIN_CONFIDENCE)
    detection_counts = Counter({frame_id: len(items) for frame_id, items in detections_by_frame.items()})

    if RENDER_FULL_VIDEO:
        start_frame = 0
        end_frame = frame_count
        window_score = sum(detection_counts.values())
    else:
        window_size = int(round(WINDOW_SECONDS * fps))
        best_start = 0
        window_score = -1
        max_start = max(0, frame_count - window_size)
        candidate_starts = range(0, max_start + 1, max(1, int(fps)))
        for start in candidate_starts:
            end = start + window_size
            score = sum(count for frame_id, count in detection_counts.items() if start <= frame_id < end)
            if score > window_score:
                best_start = start
                window_score = score
        start_frame = best_start
        end_frame = min(frame_count, start_frame + window_size)

    class_counts = Counter()
    for detections in detections_by_frame.values():
        for det in detections:
            class_counts[det["class_name"]] += 1

    render_jobs.append(
        {
            **record,
            "fps": fps,
            "frame_count": frame_count,
            "width": width,
            "height": height,
            "detections_by_frame": detections_by_frame,
            "start_frame": start_frame,
            "end_frame": end_frame,
            "window_score": window_score,
            "class_counts": dict(class_counts),
        }
    )

[
    {
        "full_id": job["full_id"],
        "fps": job["fps"],
        "frame_count": job["frame_count"],
        "size": (job["width"], job["height"]),
        "frames_with_detections": len(job["detections_by_frame"]),
        "total_detections": sum(job["class_counts"].values()),
        "class_counts": job["class_counts"],
        "start_frame": job["start_frame"],
        "end_frame": job["end_frame"],
    }
    for job in render_jobs
]

## Render Video

In [ ]:
def draw_detections(frame, detections, frame_id, full_id):
    detections = sorted(detections, key=lambda item: item["confidence"], reverse=True)[:MAX_BOXES_PER_FRAME]

    for det in detections:
        class_name = det["class_name"]
        color = CLASS_COLORS.get(class_name, DEFAULT_COLOR)
        text_color = (10, 10, 10)

        x1 = max(0, min(frame.shape[1] - 1, int(round(det["x1"]))))
        y1 = max(0, min(frame.shape[0] - 1, int(round(det["y1"]))))
        x2 = max(0, min(frame.shape[1] - 1, int(round(det["x2"]))))
        y2 = max(0, min(frame.shape[0] - 1, int(round(det["y2"]))))

        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 3)
        label = f"{class_name} {det['confidence']:.2f}"
        (label_w, label_h), baseline = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.7, 2)
        label_y1 = max(0, y1 - label_h - baseline - 8)
        label_y2 = label_y1 + label_h + baseline + 8
        cv2.rectangle(frame, (x1, label_y1), (min(frame.shape[1] - 1, x1 + label_w + 12), label_y2), color, -1)
        cv2.putText(frame, label, (x1 + 6, label_y2 - baseline - 4), cv2.FONT_HERSHEY_SIMPLEX, 0.7, text_color, 2, cv2.LINE_AA)

    overlay = f"{full_id} | frame {frame_id} | boxes {len(detections)}"
    cv2.putText(frame, overlay, (24, 42), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 255, 255), 3, cv2.LINE_AA)
    cv2.putText(frame, overlay, (24, 42), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (20, 20, 20), 1, cv2.LINE_AA)
    cv2.putText(frame, "left_hand", (24, 82), cv2.FONT_HERSHEY_SIMPLEX, 0.8, CLASS_COLORS["left_hand"], 2, cv2.LINE_AA)
    cv2.putText(frame, "right_hand", (24, 116), cv2.FONT_HERSHEY_SIMPLEX, 0.8, CLASS_COLORS["right_hand"], 2, cv2.LINE_AA)
    return frame


OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
output_video_paths = []

for job in render_jobs:
    print(f"Rendering {job['full_id']} -> {job['output_video_path']}")
    cap = cv2.VideoCapture(str(job["video_path"]))
    cap.set(cv2.CAP_PROP_POS_FRAMES, job["start_frame"])

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(
        str(job["output_video_path"]),
        fourcc,
        job["fps"],
        (job["width"], job["height"]),
    )
    if not writer.isOpened():
        raise RuntimeError(f"Could not create output video: {job['output_video_path']}")

    frame_id = job["start_frame"]
    while frame_id < job["end_frame"]:
        ok, frame = cap.read()
        if not ok:
            break
        draw_detections(frame, job["detections_by_frame"].get(frame_id, []), frame_id, job["full_id"])
        writer.write(frame)
        frame_id += 1

    cap.release()
    writer.release()
    output_video_paths.append(job["output_video_path"])

[str(path) for path in output_video_paths]

In [ ]:
for path in output_video_paths:
    print(path)
    display(Video(str(path), embed=False, html_attributes="controls muted"))

[str(path) for path in output_video_paths]

## Detection Summary Table

In [ ]:
import os
os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")

import pandas as pd
import matplotlib.pyplot as plt


def get_video_frame_count(full_id):
    participant_id = participant_from_full_id(full_id)
    path = VIDEO_ROOT / participant_id / f"{full_id}_world_camera.mp4"
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        return 0
    frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    return frames


rows = []
for summary in summaries:
    full_id = summary["full_id"]
    frames = get_video_frame_count(full_id)
    rows.append(
        {
            "full_id": full_id,
            "frames": frames,
            "frames_with_detection": summary["frames_with_detection"],
            "frame_detection_rate": summary["frames_with_detection"] / frames if frames else 0.0,
            "detections": summary["detections"],
            "left_hand": summary["left_hand"],
            "right_hand": summary["right_hand"],
            "left_rate": summary["left_hand"] / summary["detections"] if summary["detections"] else 0.0,
            "right_rate": summary["right_hand"] / summary["detections"] if summary["detections"] else 0.0,
        }
    )

rate_df = pd.DataFrame(rows).sort_values("frame_detection_rate", ascending=False).reset_index(drop=True)
display(
    rate_df.style.format(
        {
            "frame_detection_rate": "{:.1%}",
            "left_rate": "{:.1%}",
            "right_rate": "{:.1%}",
        }
    ).background_gradient(subset=["frame_detection_rate"], cmap="Greens")
)

plot_df = rate_df.head(20).sort_values("frame_detection_rate")
fig, ax = plt.subplots(figsize=(12, max(5, 0.35 * len(plot_df))))
ax.barh(plot_df["full_id"], plot_df["frame_detection_rate"], color="#2ca25f")
ax.set_xlim(0, 1)
ax.set_xlabel("frames with left/right hand detection")
ax.set_title("Left/right hand YOLO coverage")
ax.xaxis.set_major_formatter(lambda value, pos: f"{value:.0%}")
ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()